In [ ]:
# Input: MD trajectory (structure.gro + traj.xtc)
# Output: ProLIF interaction fingerprint (prolif.pkl) and interaction probability barplot
import MDAnalysis as mda
from MDAnalysis.analysis.rms import rmsd
from MDAnalysis.analysis import align
import matplotlib.pyplot as plt
import prolif
import prolif.plotting.network
import pickle

In [ ]:
u = mda.Universe('structure.gro', 'traj.xtc')

In [ ]:
#help(mda.topology.guessers)

guessed_elements = mda.topology.guessers.guess_types(u.atoms.names)
    
u.add_TopologyAttr('elements', guessed_elements)

In [ ]:

prot = u.select_atoms('protein')
dca = u.select_atoms('resname DCA')
prot.atoms.guess_bonds()
dca.atoms.guess_bonds()
fp = prolif.Fingerprint()
fp.run(u.trajectory, dca, prot)

dca = dca.convert_to('RDKIT')

dca = prolif.Molecule(dca)

In [ ]:
df = fp.to_dataframe()
with open('prolif.pkl', 'wb') as handle:
    pickle.dump(df, handle, protocol=pickle.HIGHEST_PROTOCOL)

#with open('filename.pickle', 'rb') as handle:
    #b = pickle.load(handle)
display(df)

In [ ]:
import pandas as pd
import seaborn as sns
resids = []
resnames = []
probs = []
interactions = []

for column in df.columns:
    prob = df[column].value_counts(normalize=True)
    probs.append(prob.loc[True])
    resnames.append(column[1])
    resids.append(column[1][3:])
    interactions.append(column[2])

out_df = pd.DataFrame({'Resids': resids, 'Resnames': resnames, 'Probs': probs, 'Interactions': interactions})
display(out_df)
filtered_df = out_df.loc[out_df['Probs'] > 0.00050]

hue_order = ['Hydrophobic', 'VdWContact', 'HBAcceptor', 'HBDonor']
ax = sns.barplot(data=out_df[out_df['Resnames'].isin(tuple(filtered_df['Resnames'].values))], x='Resnames', y='Probs', hue='Interactions', hue_order=hue_order)

ax.tick_params(axis='both', which='major', labelsize=6)
ax.set_ylabel('Interaction probability')
ax.set_xlabel('Resname')
ax.set_title('Protein-ligand interaction probabilities')
ax.tick_params(axis='x', labelrotation=75, labelsize='8')